In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')


RESULTS_DIR = Path('phase2/results/20260410_003735/')
print(f'Loading results from {RESULTS_DIR}...')

In [ ]:
# # Load all metrics files - if you have separate files for each mechanism, list them here, 
# otherwise just use the single merged file as in qwen models.

# metrics_files = {
#     'uniform': 'uniform/20260413_194452/metrics.json',
#     'contribution': 'contribution/20260413_195056/metrics.json',
#     'contribution_oracle': 'contribution_oracle/20260413_194804/metrics.json',
#     'counterfactual_contribution': 'counterfactual_contribution/20260413_194814/metrics.json',
#     'hybrid': 'hybrid/20260413_201229/metrics.json',
#     'stake': 'stake/20260413_194416/metrics.json',
#     'bid_to_speak': 'bid_to_speak/20260413_194423/metrics.json',
#     'free_debate': 'free_debate/20260413_200000/metrics.json',
#     'forced_sharing': 'forced_sharing/20260413_194852/metrics.json',
#     'no_comm': 'no_comm/20260413_194857/metrics.json',
# }

# Your single merged metrics file
METRICS_FILE = 'metrics.json'  # adjust path as needed

metrics_files = {name: METRICS_FILE for name in [
    'free_debate',
    'contribution',
    'forced_sharing',
    'contribution_oracle',
    'counterfactual_contribution',
    'no_comm',
    # 'hybrid',
    'bid_to_speak',
    'uniform',
    'stake',
]}

all_metrics = {}
for name, path in metrics_files.items():
    with open(RESULTS_DIR / path) as f:
        data = json.load(f)
        # Handle nested structure (metrics are inside incentive key)
        all_metrics[name] = data.get(name, data)

print(f"Loaded {len(all_metrics)} incentive mechanisms")
print("Mechanisms:", list(all_metrics.keys()))

In [ ]:
# Create summary dataframe
summary_data = []
for name, m in all_metrics.items():
    summary_data.append({
        'Incentive': name,
        'Accuracy': m.get('accuracy', np.nan),
        'Accuracy Std': m.get('std_accuracy', np.nan),
        'Decisive Surfacing Rate': m.get('decisive_surfacing_rate', np.nan),
        'Free Riding Rate': m.get('free_riding_rate', np.nan),
        'Novelty Rate': m.get('novelty_rate', np.nan),
        'Mean Disclosure Cost': m.get('mean_disclosure_cost', np.nan),
        'Mean Comm Tokens': m.get('mean_communication_tokens', np.nan),
        'Time to Decisive': m.get('time_to_decisive_surfacing', np.nan),
    })

df = pd.DataFrame(summary_data)
df = df.set_index('Incentive')
# df = df.sort_values('Accuracy', ascending=False)
df.round(3)

## Key Metrics Comparison

In [ ]:
# Accuracy comparison with confidence intervals
fig, ax = plt.subplots(figsize=(12, 6))

incentives = df.index.tolist()
accuracies = df['Accuracy'].values
stds = df['Accuracy Std'].values

# 95% CI assuming 180 samples (60 scenarios × 3 runs)
n = 180
ci = 1.96 * stds / np.sqrt(n)

colors = sns.color_palette('husl', len(incentives))
bars = ax.bar(range(len(incentives)), accuracies, yerr=ci, capsize=5, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Group Decision Accuracy by Incentive Mechanism (temp=0.0)')
ax.set_ylim(0, 1)
ax.axhline(y=accuracies.mean(), color='red', linestyle='--', label=f'Mean: {accuracies.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Decisive Feature Surfacing Rate
fig, ax = plt.subplots(figsize=(12, 6))

surfacing_rates = df['Decisive Surfacing Rate'].values
colors = sns.color_palette('husl', len(incentives))

bars = ax.bar(range(len(incentives)), surfacing_rates, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Decisive Surfacing Rate')
ax.set_title('Rate of Decisive Feature Disclosure by Incentive Mechanism')
ax.set_ylim(0, 1)
ax.axhline(y=surfacing_rates.mean(), color='red', linestyle='--', label=f'Mean: {surfacing_rates.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('surfacing_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Free Riding Rate (lower is better for cooperation)
fig, ax = plt.subplots(figsize=(12, 6))

free_riding = df['Free Riding Rate'].values
colors = sns.color_palette('husl', len(incentives))

bars = ax.bar(range(len(incentives)), free_riding, color=colors, edgecolor='black')

ax.set_xticks(range(len(incentives)))
ax.set_xticklabels(incentives, rotation=45, ha='right')
ax.set_ylabel('Free Riding Rate')
ax.set_title('Free Riding Rate by Incentive Mechanism (Lower = More Cooperation)')
ax.set_ylim(0, 1)
ax.axhline(y=free_riding.mean(), color='red', linestyle='--', label=f'Mean: {free_riding.mean():.3f}')
ax.legend()

plt.tight_layout()
plt.savefig('free_riding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Multi-Metric Radar Chart

In [ ]:
# Radar chart for top 5 mechanisms
from math import pi

# Select metrics and normalize
metrics_for_radar = ['Accuracy', 'Decisive Surfacing Rate', 'Novelty Rate']
# Invert free riding (1 - rate) so higher is better
df_radar = df[metrics_for_radar].copy()
df_radar['Cooperation Rate'] = 1 - df['Free Riding Rate']

# Normalize to 0-1
df_radar_norm = (df_radar - df_radar.min()) / (df_radar.max() - df_radar.min())

# Top 5 by accuracy
top5 = df_radar_norm.head(5)

categories = list(top5.columns)
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for i, (idx, row) in enumerate(top5.iterrows()):
    values = row.values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=idx)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_title('Top 5 Incentive Mechanisms - Multi-Metric Comparison', size=14, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Tests

In [ ]:
# Compare top mechanism vs baseline (uniform)
baseline = 'uniform'
top_mech = df.index[0]  # highest accuracy

print(f"Comparing {top_mech} vs {baseline}")
print(f"\n{top_mech}:")
print(f"  Accuracy: {all_metrics[top_mech]['accuracy']:.3f} ± {all_metrics[top_mech]['std_accuracy']:.3f}")
print(f"\n{baseline}:")
print(f"  Accuracy: {all_metrics[baseline]['accuracy']:.3f} ± {all_metrics[baseline]['std_accuracy']:.3f}")

# Cohen's d effect size (using pooled std)
m1, s1 = all_metrics[top_mech]['accuracy'], all_metrics[top_mech]['std_accuracy']
m2, s2 = all_metrics[baseline]['accuracy'], all_metrics[baseline]['std_accuracy']
pooled_std = np.sqrt((s1**2 + s2**2) / 2)
cohens_d = (m1 - m2) / pooled_std if pooled_std > 0 else 0

print(f"\nEffect Size (Cohen's d): {cohens_d:.3f}")
if abs(cohens_d) < 0.2:
    print("  Interpretation: Negligible effect")
elif abs(cohens_d) < 0.5:
    print("  Interpretation: Small effect")
elif abs(cohens_d) < 0.8:
    print("  Interpretation: Medium effect")
else:
    print("  Interpretation: Large effect")

In [ ]:
# Summary statistics table
print("\n" + "="*80)
print("SUMMARY: INCENTIVE MECHANISM COMPARISON (Temperature = 0.0)")
print("="*80)
print(f"\nBest Accuracy: {df.index[0]} ({df['Accuracy'].iloc[0]:.1%})")
print(f"Worst Accuracy: {df.index[-1]} ({df['Accuracy'].iloc[-1]:.1%})")
print(f"\nHighest Surfacing Rate: {df.sort_values('Decisive Surfacing Rate', ascending=False).index[0]}")
print(f"Lowest Free Riding: {df.sort_values('Free Riding Rate').index[0]}")

print("\n" + "-"*80)
print(df[['Accuracy', 'Decisive Surfacing Rate', 'Free Riding Rate']].round(3).to_string())

## Domain-Specific Analysis

In [ ]:
# Domain accuracy heatmap
domain_data = {}
for name, m in all_metrics.items():
    if 'per_domain' in m:
        domain_data[name] = m['per_domain']

if domain_data:
    domain_df = pd.DataFrame(domain_data)
    
    # Sort by mean accuracy across mechanisms
    domain_df['mean'] = domain_df.mean(axis=1)
    domain_df = domain_df.sort_values('mean', ascending=False)
    domain_df = domain_df.drop('mean', axis=1)
    
    plt.figure(figsize=(14, 16))
    sns.heatmap(domain_df, annot=True, fmt='.2f', cmap='RdYlGn', 
                vmin=0, vmax=1, linewidths=0.5)
    plt.title('Accuracy by Domain and Incentive Mechanism')
    plt.xlabel('Incentive Mechanism')
    plt.ylabel('Domain')
    plt.tight_layout()
    plt.savefig('domain_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No per-domain data available")

In [ ]:
# Save summary to CSV
df.to_csv('incentive_comparison_summary.csv')
print("Summary saved to incentive_comparison_summary.csv")

## Core Research Questions

### 1. Free-Rider Problem: Does Uniform Reward Lead to Under-Disclosure?
The central hypothesis: uniform incentives + costly disclosure = rational under-revelation of decisive information.

In [ ]:
# Key comparison: Uniform vs Contribution-based incentives
# This tests the core hypothesis about free-riding

uniform_acc = all_metrics['uniform']['accuracy']
contribution_acc = all_metrics['contribution']['accuracy']
contribution_oracle_acc = all_metrics['contribution_oracle']['accuracy']

uniform_surfacing = all_metrics['uniform']['decisive_surfacing_rate']
contribution_surfacing = all_metrics['contribution']['decisive_surfacing_rate']
contribution_oracle_surfacing = all_metrics['contribution_oracle']['decisive_surfacing_rate']

print("="*70)
print("HYPOTHESIS TEST: Do contribution-based incentives reduce free-riding?")
print("="*70)

print(f"\n{'Metric':<30} {'Uniform':<15} {'Contribution':<15} {'Oracle':<15}")
print("-"*70)
print(f"{'Accuracy':<30} {uniform_acc:.3f}          {contribution_acc:.3f}          {contribution_oracle_acc:.3f}")
print(f"{'Decisive Surfacing Rate':<30} {uniform_surfacing:.3f}          {contribution_surfacing:.3f}          {contribution_oracle_surfacing:.3f}")
print(f"{'Free Riding Rate':<30} {all_metrics['uniform']['free_riding_rate']:.3f}          {all_metrics['contribution']['free_riding_rate']:.3f}          {all_metrics['contribution_oracle']['free_riding_rate']:.3f}")

# Improvement percentages
acc_improvement = (contribution_acc - uniform_acc) / uniform_acc * 100
surfacing_improvement = (contribution_surfacing - uniform_surfacing) / uniform_surfacing * 100 if uniform_surfacing > 0 else 0

print(f"\n📈 Contribution vs Uniform:")
print(f"   Accuracy improvement: {acc_improvement:+.1f}%")
print(f"   Surfacing improvement: {surfacing_improvement:+.1f}%")

### 2. Disclosure Calibration: Are Agents Cost-Sensitive?
Do agents share high-value features and withhold noise? Or do they free-ride?

In [ ]:
# Surfacing rate by feature cost (from metrics)
# This shows if high-cost decisive features are under-disclosed

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Surfacing by cost for each mechanism
ax1 = axes[0]
for name, m in all_metrics.items():
    if 'surfacing_by_cost' in m and m['surfacing_by_cost']:
        costs = sorted([int(k) for k in m['surfacing_by_cost'].keys()])
        rates = [m['surfacing_by_cost'][str(c)] for c in costs]
        ax1.plot(costs, rates, 'o-', label=name, alpha=0.7)

ax1.set_xlabel('Feature Disclosure Cost')
ax1.set_ylabel('Surfacing Rate')
ax1.set_title('Disclosure Rate vs Feature Cost\n(Lower surfacing at high cost = cost-sensitivity)')
ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax1.set_xticks([1, 2, 3, 4, 5])
ax1.grid(True, alpha=0.3)

# Plot 2: Selective Disclosure Index comparison
ax2 = axes[1]
sdi_data = [(name, m.get('selective_disclosure_index', 0)) for name, m in all_metrics.items()]
sdi_data.sort(key=lambda x: x[1], reverse=True)
names, sdis = zip(*sdi_data)

colors = sns.color_palette('husl', len(names))
ax2.barh(range(len(names)), sdis, color=colors)
ax2.set_yticks(range(len(names)))
ax2.set_yticklabels(names)
ax2.set_xlabel('Selective Disclosure Index')
ax2.set_title('Selective Disclosure Index\n(Higher = More focused on decisive features)')
ax2.axvline(x=np.mean(sdis), color='red', linestyle='--', label=f'Mean: {np.mean(sdis):.3f}')

plt.tight_layout()
plt.savefig('disclosure_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

### 3. Misleading Feature Influence
How often do low-cost misleading features dominate discussion and steer the group wrong?

In [ ]:
# Misleading feature analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Misleading before wrong rate
misleading_data = []
for name, m in all_metrics.items():
    misleading_data.append({
        'Incentive': name,
        'Misleading Before Wrong': m.get('misleading_before_wrong_rate', 0),
        'Misleading Preceded Decisive': m.get('misleading_preceded_decisive_rate', 0),
    })

misleading_df = pd.DataFrame(misleading_data).set_index('Incentive')
misleading_df = misleading_df.sort_values('Misleading Before Wrong', ascending=False)

ax1 = axes[0]
x = np.arange(len(misleading_df))
width = 0.35
bars1 = ax1.bar(x - width/2, misleading_df['Misleading Before Wrong'], width, label='Misleading Before Wrong', color='coral')
bars2 = ax1.bar(x + width/2, misleading_df['Misleading Preceded Decisive'], width, label='Misleading Preceded Decisive', color='steelblue')

ax1.set_ylabel('Rate')
ax1.set_title('Misleading Feature Influence\n(Lower = Better noise filtering)')
ax1.set_xticks(x)
ax1.set_xticklabels(misleading_df.index, rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0, max(misleading_df.max()) * 1.2)

# Plot 2: Decisive holder disclosure rate vs non-holder
ax2 = axes[1]
holder_data = []
for name, m in all_metrics.items():
    holder_data.append({
        'Incentive': name,
        'Decisive Holder Rate': m.get('decisive_holder_disclosure_rate', 0),
        'Non-Holder Rate': m.get('non_holder_disclosure_rate', 0),
    })

holder_df = pd.DataFrame(holder_data).set_index('Incentive')
holder_df = holder_df.sort_values('Decisive Holder Rate', ascending=False)

x = np.arange(len(holder_df))
bars1 = ax2.bar(x - width/2, holder_df['Decisive Holder Rate'], width, label='Decisive Holder', color='green')
bars2 = ax2.bar(x + width/2, holder_df['Non-Holder Rate'], width, label='Non-Holder', color='gray')

ax2.set_ylabel('Disclosure Rate')
ax2.set_title('Who Discloses?\n(Decisive holders should disclose more)')
ax2.set_xticks(x)
ax2.set_xticklabels(holder_df.index, rotation=45, ha='right')
ax2.legend()

plt.tight_layout()
plt.savefig('misleading_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. Cost-Effectiveness Analysis
Accuracy per communication cost - mapping the accuracy-cost frontier

In [ ]:
# Cost-effectiveness: Accuracy vs Communication Cost
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gather data
cost_eff_data = []
for name, m in all_metrics.items():
    acc = m.get('accuracy', 0)
    tokens = m.get('mean_communication_tokens', 1)
    disclosure_cost = m.get('mean_disclosure_cost', 0)
    cost_eff_data.append({
        'Incentive': name,
        'Accuracy': acc,
        'Mean Tokens': tokens,
        'Mean Disclosure Cost': disclosure_cost,
        'Accuracy per 1K Tokens': acc / (tokens / 1000) if tokens > 0 else 0,
    })

cost_df = pd.DataFrame(cost_eff_data)

# Plot 1: Accuracy vs Token Cost (Pareto frontier)
ax1 = axes[0]
colors = sns.color_palette('husl', len(cost_df))
for i, row in cost_df.iterrows():
    ax1.scatter(row['Mean Tokens'], row['Accuracy'], s=150, c=[colors[i]], edgecolor='black', zorder=5)
    ax1.annotate(row['Incentive'], (row['Mean Tokens'], row['Accuracy']), 
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

ax1.set_xlabel('Mean Communication Tokens')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy-Cost Frontier\n(Upper-left = most efficient)')
ax1.grid(True, alpha=0.3)

# Highlight Pareto-optimal points
sorted_df = cost_df.sort_values('Mean Tokens')
pareto_acc = 0
pareto_points = []
for _, row in sorted_df.iterrows():
    if row['Accuracy'] > pareto_acc:
        pareto_points.append((row['Mean Tokens'], row['Accuracy']))
        pareto_acc = row['Accuracy']

if pareto_points:
    px, py = zip(*pareto_points)
    ax1.plot(px, py, 'r--', alpha=0.5, label='Pareto frontier')
    ax1.legend()

# Plot 2: Cost-effectiveness ranking
ax2 = axes[1]
cost_df_sorted = cost_df.sort_values('Accuracy per 1K Tokens', ascending=True)
colors = sns.color_palette('husl', len(cost_df_sorted))
ax2.barh(range(len(cost_df_sorted)), cost_df_sorted['Accuracy per 1K Tokens'], color=colors)
ax2.set_yticks(range(len(cost_df_sorted)))
ax2.set_yticklabels(cost_df_sorted['Incentive'])
ax2.set_xlabel('Accuracy per 1K Tokens')
ax2.set_title('Cost-Effectiveness Ranking\n(Higher = more efficient)')

plt.tight_layout()
plt.savefig('cost_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

# Print cost-effectiveness table
print("\nCost-Effectiveness Summary:")
print(cost_df.sort_values('Accuracy per 1K Tokens', ascending=False).to_string(index=False))

### 5. Time to Decisive Information
How quickly does decisive information surface? Earlier is better.

In [ ]:
# Time to decisive surfacing
time_data = [(name, m.get('time_to_decisive_surfacing', np.nan)) for name, m in all_metrics.items()]
time_df = pd.DataFrame(time_data, columns=['Incentive', 'Time to Decisive'])
time_df = time_df.dropna().sort_values('Time to Decisive')

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('husl', len(time_df))
bars = ax.barh(range(len(time_df)), time_df['Time to Decisive'], color=colors)
ax.set_yticks(range(len(time_df)))
ax.set_yticklabels(time_df['Incentive'])
ax.set_xlabel('Mean Round When Decisive Feature Surfaces')
ax.set_title('Time to Decisive Information Surfacing\n(Lower = faster revelation)')
ax.axvline(x=time_df['Time to Decisive'].mean(), color='red', linestyle='--', 
           label=f"Mean: {time_df['Time to Decisive'].mean():.2f}")
ax.legend()

plt.tight_layout()
plt.savefig('time_to_decisive.png', dpi=150, bbox_inches='tight')
plt.show()

### 6. Mechanism Categories Comparison
Group mechanisms by type: Market-based, Contribution-based, Baselines

In [ ]:
# Group mechanisms by category
mechanism_categories = {
    'Baseline': ['uniform', 'free_debate', 'no_comm'],
    'Contribution-Based': ['contribution', 'contribution_oracle', 'counterfactual_contribution'],
    'Market-Based': ['stake', 'bid_to_speak'],
    'Hybrid/Forced': ['hybrid', 'forced_sharing'],
}

category_stats = []
for cat, mechs in mechanism_categories.items():
    cat_accs = [all_metrics[m]['accuracy'] for m in mechs if m in all_metrics]
    cat_surfacing = [all_metrics[m]['decisive_surfacing_rate'] for m in mechs if m in all_metrics]
    cat_free_riding = [all_metrics[m]['free_riding_rate'] for m in mechs if m in all_metrics]
    
    category_stats.append({
        'Category': cat,
        'Mean Accuracy': np.mean(cat_accs),
        'Std Accuracy': np.std(cat_accs),
        'Mean Surfacing': np.mean(cat_surfacing),
        'Mean Free Riding': np.mean(cat_free_riding),
        'N': len(cat_accs),
    })

cat_df = pd.DataFrame(category_stats)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy by category
ax1 = axes[0]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
bars = ax1.bar(cat_df['Category'], cat_df['Mean Accuracy'], yerr=cat_df['Std Accuracy'], 
               capsize=5, color=colors, edgecolor='black')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy by Mechanism Category')
ax1.set_ylim(0, 1)
ax1.tick_params(axis='x', rotation=15)

# Surfacing by category
ax2 = axes[1]
ax2.bar(cat_df['Category'], cat_df['Mean Surfacing'], color=colors, edgecolor='black')
ax2.set_ylabel('Decisive Surfacing Rate')
ax2.set_title('Decisive Surfacing by Category')
ax2.set_ylim(0, 1)
ax2.tick_params(axis='x', rotation=15)

# Free riding by category
ax3 = axes[2]
ax3.bar(cat_df['Category'], cat_df['Mean Free Riding'], color=colors, edgecolor='black')
ax3.set_ylabel('Free Riding Rate')
ax3.set_title('Free Riding by Category')
ax3.set_ylim(0, 1)
ax3.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('category_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCategory Summary:")
print(cat_df.round(3).to_string(index=False))

### 7. Final Paper-Ready Summary Table

In [ ]:
# Paper-ready LaTeX table
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Incentive Mechanism Comparison (Temperature = 0.0, N=180 per mechanism)}")
print("\\begin{tabular}{lcccccc}")
print("\\toprule")
print("Mechanism & Accuracy & Surfacing & Free-Riding & SDI & Time & Tokens \\\\")
print("\\midrule")

for name in df.index:
    m = all_metrics[name]
    acc = m.get('accuracy', 0)
    surf = m.get('decisive_surfacing_rate', 0)
    free = m.get('free_riding_rate', 0)
    sdi = m.get('selective_disclosure_index', 0)
    time_val = m.get('time_to_decisive_surfacing', 0)
    tokens = m.get('mean_communication_tokens', 0)
    
    escaped_name = name.replace('_', '\\_')
    print(f"{escaped_name} & {acc:.3f} & {surf:.3f} & {free:.3f} & {sdi:.3f} & {time_val:.2f} & {tokens:.0f} \\\\")

print("\\bottomrule")
print("\\end{tabular}")
print("\\label{tab:incentive_comparison}")
print("\\end{table}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print(f"\n1. Best Accuracy: {df.index[0]} ({df['Accuracy'].iloc[0]:.1%})")
print(f"2. Best Surfacing: {df.sort_values('Decisive Surfacing Rate', ascending=False).index[0]}")
print(f"3. Lowest Free-Riding: {df.sort_values('Free Riding Rate').index[0]}")
print(f"\n4. Contribution-based vs Uniform improvement:")
acc_diff = (all_metrics['contribution']['accuracy'] - all_metrics['uniform']['accuracy'])*100
surf_diff = (all_metrics['contribution']['decisive_surfacing_rate'] - all_metrics['uniform']['decisive_surfacing_rate'])*100
print(f"   Accuracy: {acc_diff:+.1f}pp")
print(f"   Surfacing: {surf_diff:+.1f}pp")

---

## 8. Error Analysis: Qualitative Behavioral Patterns

Aggregate metrics tell us *that* mechanisms differ. This section asks **why** — by examining individual scenario trajectories to identify recurring failure and success modes at the semantic level.

The analysis proceeds in four steps:
1. **Outcome matrix** — per-scenario × per-mechanism correctness grid
2. **Divergent scenarios** — cases where mechanisms disagree (some right, some wrong on the same scenario)
3. **Transcript loading** — pull raw deliberation text for divergent cases
4. **Semantic pattern classification** — use the Claude API to label each case with a behavior category, then aggregate into a taxonomy


In [ ]:
# ── 8.1  Load per-scenario trajectory data ────────────────────────────────
# Results are stored either as one merged file (same dir as metrics.json)
# or as per-mechanism subdirs.  We try both layouts.

import json, os
from pathlib import Path

RESULTS_DIR = Path('phase2/results/20260410_003735/')   # same as above

# Attempt 1: single merged results file alongside metrics.json
merged_candidates = [
    RESULTS_DIR / 'results.json',
    RESULTS_DIR / 'scenarios_results.json',
    RESULTS_DIR / 'full_results.json',
]
raw_results = {}          # {mechanism: [scenario_result, ...]}

for cand in merged_candidates:
    if cand.exists():
        with open(cand) as f:
            data = json.load(f)
        # If top-level keys are mechanism names, use directly
        if isinstance(data, dict) and set(data.keys()) & set(all_metrics.keys()):
            raw_results = data
            print(f"Loaded merged results from {cand}")
        break

# Attempt 2: per-mechanism subdirs  → results.json / scenarios.json
if not raw_results:
    for mech in all_metrics:
        for fname in ['results.json', 'scenarios.json']:
            candidate = RESULTS_DIR / mech / fname
            if candidate.exists():
                with open(candidate) as f:
                    raw_results[mech] = json.load(f)
                break
    if raw_results:
        print(f"Loaded per-mechanism results for: {list(raw_results.keys())}")

if not raw_results:
    print("⚠  No raw trajectory files found.  Error analysis will use metrics-level data only.")
    print("   Expected: phase2/results/<run>/results.json  or  phase2/results/<run>/<mech>/results.json")
else:
    # Normalise: each mechanism value should be a list of scenario dicts
    for mech, val in raw_results.items():
        if isinstance(val, dict) and 'scenarios' in val:
            raw_results[mech] = val['scenarios']
    sample_mech = next(iter(raw_results))
    sample_scenario = raw_results[sample_mech][0] if raw_results[sample_mech] else {}
    print(f"Sample scenario keys: {list(sample_scenario.keys())[:10]}")
    print(f"Mechanisms loaded: {list(raw_results.keys())}")
    print(f"Scenarios per mechanism (sample): {len(raw_results[sample_mech])}")


### 8.2  Cross-Mechanism Outcome Matrix

Each cell shows whether mechanism M answered scenario S correctly (✓) or not (✗).
Scenarios where mechanisms **disagree** are the most informative for error analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Build outcome matrix ───────────────────────────────────────────────────
# If raw_results is available, derive correctness per scenario.
# Fall back to synthetic illustration using aggregate accuracy if not.

CORRECT_KEY   = 'correct'       # bool field in each scenario dict
SCENARIO_KEY  = 'scenario_id'   # or 'id', 'scenario_idx'

def _find_key(d, candidates, default=None):
    for k in candidates:
        if k in d:
            return d[k]
    return default

if raw_results:
    # Collect all scenario IDs across mechanisms
    all_ids = set()
    for mech, scenarios in raw_results.items():
        for s in scenarios:
            sid = _find_key(s, ['scenario_id', 'id', 'idx', 'scenario_idx'], None)
            if sid is not None:
                all_ids.add(sid)

    all_ids = sorted(all_ids)
    outcome_rows = {}
    for mech, scenarios in raw_results.items():
        row = {}
        for s in scenarios:
            sid = _find_key(s, ['scenario_id', 'id', 'idx', 'scenario_idx'])
            correct = _find_key(s, ['correct', 'is_correct', 'accuracy', 'decision_correct'])
            if sid is not None and correct is not None:
                row[sid] = bool(correct)
        outcome_rows[mech] = row

    outcome_df = pd.DataFrame(outcome_rows, index=all_ids).T   # shape: mechs × scenarios
    outcome_df.index.name = 'mechanism'
    print(f"Outcome matrix shape: {outcome_df.shape}  (mechanisms × scenarios)")
else:
    print("Using simulated outcome matrix for illustration (no raw trajectory data found).")
    mechs = list(all_metrics.keys())
    n_scenarios = 60
    rng = np.random.default_rng(42)
    sim_data = {
        mech: rng.random(n_scenarios) < all_metrics[mech]['accuracy']
        for mech in mechs
    }
    outcome_df = pd.DataFrame(sim_data).T
    outcome_df.index.name = 'mechanism'

# ── Visualise ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(min(24, max(10, len(outcome_df.columns) // 2)), 5))
cmap = sns.color_palette(['#d62728', '#2ca02c'], as_cmap=False)
binary_data = outcome_df.astype(float).values
sns.heatmap(binary_data, ax=ax, cmap=['#ffaaaa', '#aaffaa'],
            cbar=False, linewidths=0, xticklabels=False,
            yticklabels=outcome_df.index.tolist())
ax.set_title('Outcome Matrix  (green = correct, red = wrong)', fontsize=13)
ax.set_xlabel('Scenarios')
ax.set_ylabel('Mechanism')
plt.tight_layout()
plt.savefig('error_outcome_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("\nPer-mechanism accuracy from matrix:")
for mech in outcome_df.index:
    vals = outcome_df.loc[mech].dropna()
    print(f"  {mech:<35} {vals.mean():.3f}  ({int(vals.sum())}/{len(vals)})")


### 8.3  Divergent Scenarios

A *divergent scenario* is one where at least one mechanism gets it right and at least one gets it wrong.
These are the cases worth studying — they reveal what each mechanism adds or misses.

We also flag **unanimous failures** (all mechanisms wrong) which point to inherent scenario difficulty.


In [ ]:
# ── Classify scenarios by outcome pattern ─────────────────────────────────
scenario_summary = []
for sid in outcome_df.columns:
    col = outcome_df[sid].dropna()
    n_correct = col.sum()
    n_total   = len(col)
    scenario_summary.append({
        'scenario_id': sid,
        'n_correct':   n_correct,
        'n_wrong':     n_total - n_correct,
        'n_mechs':     n_total,
        'pct_correct': n_correct / n_total if n_total else np.nan,
        'type': (
            'unanimous_correct'  if n_correct == n_total else
            'unanimous_failure'  if n_correct == 0        else
            'majority_correct'   if n_correct > n_total / 2 else
            'majority_wrong'
        ),
    })

scen_df = pd.DataFrame(scenario_summary)

# ── Summary counts ─────────────────────────────────────────────────────────
type_counts = scen_df['type'].value_counts()
print("Scenario outcome classification:")
print(type_counts.to_string())
print(f"\nDivergent scenarios (not unanimous): "
      f"{len(scen_df[~scen_df['type'].str.startswith('unanimous')])}")

# ── Plot distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: category breakdown
color_map = {
    'unanimous_correct': '#2ca02c',
    'majority_correct':  '#98df8a',
    'majority_wrong':    '#ff9896',
    'unanimous_failure': '#d62728',
}
cats = ['unanimous_correct', 'majority_correct', 'majority_wrong', 'unanimous_failure']
counts = [type_counts.get(c, 0) for c in cats]
colors = [color_map[c] for c in cats]
axes[0].bar(cats, counts, color=colors, edgecolor='black')
axes[0].set_xticklabels([c.replace('_', '\n') for c in cats], fontsize=8)
axes[0].set_ylabel('# Scenarios')
axes[0].set_title('Scenario Difficulty Distribution')

# Right: histogram of % mechanisms correct
axes[1].hist(scen_df['pct_correct'].dropna(), bins=20, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Fraction of mechanisms correct')
axes[1].set_ylabel('# Scenarios')
axes[1].set_title('How Many Mechanisms Solve Each Scenario?')
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.7, label='50%')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('error_scenario_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Store divergent set for later
divergent_ids = scen_df[~scen_df['type'].str.startswith('unanimous')]['scenario_id'].tolist()
print(f"\n{len(divergent_ids)} divergent scenario IDs stored → `divergent_ids`")


### 8.4  Retrieve Deliberation Transcripts for Divergent Cases

For each divergent scenario we pull:
- the **winning mechanism** transcript (first mechanism that answered correctly)
- the **losing mechanism** transcript (highest-accuracy mechanism that got it wrong)

This gives Claude API the minimal contrast needed to explain the difference.


In [ ]:
# ── Build transcript pairs for divergent scenarios ─────────────────────────
# Mechanism ordering by overall accuracy (best first)
mech_accuracy_rank = sorted(
    all_metrics.keys(),
    key=lambda m: all_metrics[m].get('accuracy', 0),
    reverse=True,
)

def extract_transcript(scenario_dict):
    """Pull deliberation text from a scenario result dict.
    Tries common key names used across codebase variants."""
    for key in ['transcript', 'deliberation', 'messages', 'conversation', 'history']:
        val = scenario_dict.get(key)
        if val:
            if isinstance(val, str):
                return val
            if isinstance(val, list):
                parts = []
                for msg in val:
                    if isinstance(msg, dict):
                        role    = msg.get('role', msg.get('agent', ''))
                        content = msg.get('content', msg.get('text', ''))
                        parts.append(f"[{role}]: {content}")
                    elif isinstance(msg, str):
                        parts.append(msg)
                return '\n'.join(parts)
    return None

transcript_pairs = []   # list of dicts for Claude API analysis

if raw_results:
    # Index: {mech: {scenario_id: scenario_dict}}
    idx = {}
    for mech, scenarios in raw_results.items():
        idx[mech] = {}
        for s in scenarios:
            sid = _find_key(s, ['scenario_id', 'id', 'idx', 'scenario_idx'])
            if sid is not None:
                idx[mech][sid] = s

    for sid in divergent_ids[:40]:   # cap at 40 pairs for API cost
        col   = outcome_df[sid].dropna()
        right = [m for m in mech_accuracy_rank if col.get(m) == True]
        wrong = [m for m in mech_accuracy_rank if col.get(m) == False]
        if not right or not wrong:
            continue
        winner = right[0]
        loser  = wrong[0]
        win_s  = idx.get(winner, {}).get(sid)
        lose_s = idx.get(loser,  {}).get(sid)
        if win_s is None or lose_s is None:
            continue
        pair = {
            'scenario_id':    sid,
            'winner':         winner,
            'loser':          loser,
            'winner_transcript': extract_transcript(win_s),
            'loser_transcript':  extract_transcript(lose_s),
            'scenario_text':  _find_key(win_s, ['scenario', 'scenario_text', 'prompt', 'description'], ''),
            'correct_answer': _find_key(win_s, ['correct_answer', 'answer', 'ground_truth', 'label'], ''),
        }
        transcript_pairs.append(pair)

    print(f"Built {len(transcript_pairs)} transcript pairs from {len(divergent_ids)} divergent scenarios")
    if transcript_pairs:
        p = transcript_pairs[0]
        print(f"\nSample pair — scenario {p['scenario_id']}:")
        print(f"  Winner:  {p['winner']}")
        print(f"  Loser:   {p['loser']}")
        print(f"  Transcript chars (winner): {len(p['winner_transcript'] or '')}")
        print(f"  Transcript chars (loser):  {len(p['loser_transcript'] or '')}")
else:
    print("⚠  No raw trajectory data — transcript pairs unavailable.")
    print("   The next cell will use the Claude API with simulated prompts for demonstration.")
    # Populate stub pairs so the next cell can still run
    for sid in list(range(1, 11)):
        transcript_pairs.append({
            'scenario_id':       sid,
            'winner':            'contribution',
            'loser':             'uniform',
            'winner_transcript': None,
            'loser_transcript':  None,
            'scenario_text':     f'(Simulated scenario {sid} — no trajectory data available)',
            'correct_answer':    'Yes',
        })


### 8.5  Semantic Pattern Classification via Claude API

For each divergent pair we ask Claude to:
1. Read both transcripts
2. Assign one **failure pattern** to the losing mechanism's deliberation
3. Assign one **success pattern** to the winning mechanism's deliberation
4. Return a short justification

The taxonomy of patterns is defined below and kept fixed so counts are comparable.


In [ ]:
import json, re, time
import anthropic

# ── Taxonomy (keep stable across runs) ────────────────────────────────────
FAILURE_PATTERNS = [
    "FREE_RIDE_WAIT"         ,  # agents withhold, each expecting others to disclose
    "FREE_RIDE_PIGGYBACK"    ,  # agents copy already-shared info without adding own
    "PREMATURE_CONSENSUS"    ,  # group converges before decisive feature surfaces
    "MISLEADING_DOMINANCE"   ,  # low-cost misleading feature steers discussion off track
    "DECISIVE_BURIED"        ,  # decisive feature mentioned but ignored in final vote
    "WRONG_MAJORITY_CASCADE" ,  # initial wrong vote adopted by others via conformity
    "NO_DISCLOSURE_AT_ALL"   ,  # agent never reveals any private feature
    "CONFUSED_COUNTERFACTUAL",  # agent reasons incorrectly about what others know
    "OTHER"                  ,
]

SUCCESS_PATTERNS = [
    "INCENTIVE_TRIGGERED"    ,  # agent explicitly cites reward mechanism as disclosure motive
    "DECISIVE_SURFACED_EARLY",  # decisive feature shared in round 1-2, anchors discussion
    "OVERRIDE_MISLEADING"    ,  # decisive feature surfaced after initial misleading discussion
    "CORRECT_SYNTHESIS"      ,  # agent correctly integrates multiple partial signals
    "FORCED_SHARE"           ,  # mechanism forced disclosure (forced_sharing / bid_to_speak)
    "NATURAL_NORM"           ,  # agents share freely with no strategic reasoning apparent
    "OTHER"                  ,
]

SYSTEM_PROMPT = """You are a research assistant analysing multi-agent LLM deliberation transcripts.
You will be shown two transcripts for the same hidden-profile scenario:
  • WINNER transcript — the mechanism that reached the correct group answer
  • LOSER  transcript — the mechanism that reached the wrong group answer

Your task: classify the PRIMARY failure mode of the LOSER and the PRIMARY success mode of the WINNER.

Return ONLY a JSON object with exactly these keys:
{
  "failure_pattern": "<one of: FREE_RIDE_WAIT | FREE_RIDE_PIGGYBACK | PREMATURE_CONSENSUS | MISLEADING_DOMINANCE | DECISIVE_BURIED | WRONG_MAJORITY_CASCADE | NO_DISCLOSURE_AT_ALL | CONFUSED_COUNTERFACTUAL | OTHER>",
  "success_pattern": "<one of: INCENTIVE_TRIGGERED | DECISIVE_SURFACED_EARLY | OVERRIDE_MISLEADING | CORRECT_SYNTHESIS | FORCED_SHARE | NATURAL_NORM | OTHER>",
  "failure_quote": "<≤25-word verbatim excerpt from loser transcript that best illustrates the failure>",
  "success_quote": "<≤25-word verbatim excerpt from winner transcript that best illustrates the success>",
  "notes": "<1-2 sentence explanation of why the winner succeeded and the loser failed>"
}
No markdown, no preamble, only the JSON object."""


def classify_pair(pair: dict, client) -> dict | None:
    """Call Claude API to classify one transcript pair."""
    has_transcripts = bool(pair['winner_transcript'] and pair['loser_transcript'])

    if has_transcripts:
        w_preview = (pair['winner_transcript'] or '')[:3000]
        l_preview = (pair['loser_transcript']  or '')[:3000]
        user_content = (
            f"Scenario: {str(pair['scenario_text'])[:400]}\n"
            f"Correct answer: {pair['correct_answer']}\n"
            f"Winner mechanism: {pair['winner']}\n"
            f"Loser mechanism: {pair['loser']}\n\n"
            f"--- WINNER TRANSCRIPT ---\n{w_preview}\n\n"
            f"--- LOSER TRANSCRIPT ---\n{l_preview}"
        )
    else:
        # No transcripts — ask model to hypothesise based on mechanism names only
        user_content = (
            f"Scenario {pair['scenario_id']}: {str(pair['scenario_text'])[:400]}\n"
            f"Correct answer: {pair['correct_answer']}\n"
            f"The mechanism '{pair['winner']}' answered correctly; '{pair['loser']}' did not.\n"
            f"Based on the known behaviour of these mechanisms, hypothesise failure/success patterns."
        )

    try:
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=400,
            system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': user_content}],
        )
        text = response.content[0].text.strip()
        # Strip any accidental markdown fences
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL).strip()
        result = json.loads(text)
        result['scenario_id'] = pair['scenario_id']
        result['winner']      = pair['winner']
        result['loser']       = pair['loser']
        result['has_transcripts'] = has_transcripts
        return result
    except Exception as e:
        print(f"  ⚠  scenario {pair['scenario_id']}: {e}")
        return None


# ── Run classification (capped at 30 pairs to limit API cost) ──────────────
client = anthropic.Anthropic()

classified = []
pairs_to_classify = transcript_pairs[:30]
print(f"Classifying {len(pairs_to_classify)} pairs via Claude API …")

for i, pair in enumerate(pairs_to_classify):
    print(f"  {i+1:3d}/{len(pairs_to_classify)}  scenario {pair['scenario_id']}  ", end='', flush=True)
    result = classify_pair(pair, client)
    if result:
        classified.append(result)
        print(result['failure_pattern'])
    else:
        print('FAILED')
    if i < len(pairs_to_classify) - 1:
        time.sleep(0.3)   # light rate-limit buffer

print(f"\nClassified {len(classified)} pairs successfully.")


### 8.6  Pattern Taxonomy & Counts

Aggregate the per-scenario labels into a taxonomy showing which failure/success modes are most prevalent.


In [ ]:
# ── Aggregate labels ───────────────────────────────────────────────────────
classified_df = pd.DataFrame(classified)

if classified_df.empty:
    print("No classified results to display.")
else:
    fail_counts    = classified_df['failure_pattern'].value_counts()
    success_counts = classified_df['success_pattern'].value_counts()

    print("FAILURE PATTERNS (across all divergent scenarios):")    print(fail_counts.to_string())
    print("\nSUCCESS PATTERNS:")
    print(success_counts.to_string())

    # ── Taxonomy bar charts ────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    fail_palette = sns.color_palette('Reds_r', len(fail_counts))
    axes[0].barh(range(len(fail_counts)), fail_counts.values, color=fail_palette, edgecolor='black')
    axes[0].set_yticks(range(len(fail_counts)))
    axes[0].set_yticklabels(fail_counts.index, fontsize=9)
    axes[0].set_xlabel('# Scenarios')
    axes[0].set_title('Failure Pattern Taxonomy', fontsize=12)
    axes[0].invert_yaxis()

    succ_palette = sns.color_palette('Greens_r', len(success_counts))
    axes[1].barh(range(len(success_counts)), success_counts.values, color=succ_palette, edgecolor='black')
    axes[1].set_yticks(range(len(success_counts)))
    axes[1].set_yticklabels(success_counts.index, fontsize=9)
    axes[1].set_xlabel('# Scenarios')
    axes[1].set_title('Success Pattern Taxonomy', fontsize=12)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig('error_pattern_taxonomy.png', dpi=150, bbox_inches='tight')
    plt.show()


### 8.7  Failure Patterns by Mechanism

Which mechanisms trigger which failure modes? A heatmap over (mechanism, failure_pattern) shows whether
some failure modes are mechanism-specific versus domain-general.


In [ ]:
if not classified_df.empty and 'loser' in classified_df.columns:
    # Pivot: rows = loser mechanism, cols = failure pattern
    fp_pivot = (classified_df
                .groupby(['loser', 'failure_pattern'])
                .size()
                .unstack(fill_value=0))

    fig, ax = plt.subplots(figsize=(max(10, len(fp_pivot.columns) * 1.2), max(4, len(fp_pivot) * 0.8)))
    sns.heatmap(fp_pivot, annot=True, fmt='d', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'count'})
    ax.set_title('Failure Patterns by Mechanism\n(row = mechanism that failed, col = how it failed)',
                 fontsize=12)
    ax.set_xlabel('Failure Pattern')
    ax.set_ylabel('Losing Mechanism')
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig('error_mechanism_failure_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

    # And success patterns by winner
    sp_pivot = (classified_df
                .groupby(['winner', 'success_pattern'])
                .size()
                .unstack(fill_value=0))

    fig, ax = plt.subplots(figsize=(max(10, len(sp_pivot.columns) * 1.2), max(4, len(sp_pivot) * 0.8)))
    sns.heatmap(sp_pivot, annot=True, fmt='d', cmap='YlGn',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'count'})
    ax.set_title('Success Patterns by Mechanism\n(row = mechanism that won, col = how it won)',
                 fontsize=12)
    ax.set_xlabel('Success Pattern')
    ax.set_ylabel('Winning Mechanism')
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig('error_mechanism_success_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No data for mechanism-level breakdown.")


### 8.8  Illustrative Examples per Failure Mode

For each failure pattern, display the most representative case: the short quote that exemplifies the
failure and Claude's explanation of why the losing mechanism went wrong.


In [ ]:
if not classified_df.empty:
    patterns_to_show = classified_df['failure_pattern'].value_counts().head(6).index.tolist()

    for pattern in patterns_to_show:
        subset = classified_df[classified_df['failure_pattern'] == pattern]
        # Pick the example whose notes field is longest (most informative)
        best = subset.loc[subset['notes'].str.len().idxmax()]

        print(f"{'='*72}")
        print(f"FAILURE: {pattern}  ({len(subset)} cases)")
        print(f"  Scenario  : {best['scenario_id']}")
        print(f"  Winner    : {best['winner']}    Loser: {best['loser']}")
        print(f"  Loser quote : \"{best.get('failure_quote', '—')}\"")
        print(f"  Winner quote: \"{best.get('success_quote', '—')}\"")
        print(f"  Analysis  : {best['notes']}")
        print()


### 8.9  "Correct Despite Different" Paths — Mechanisms That Agree on the Answer but Disagree on How

Among scenarios where **two or more mechanisms all got the answer right**, we can still ask:
did they arrive there via different deliberation paths?

This cell uses Claude API to compare winning transcripts across mechanisms for the same scenario,
flagging cases where the *reasoning* diverged even though the *outcome* converged.
This tests whether contribution-based mechanisms surface decisive features earlier/more explicitly
even when free_debate also eventually gets there by luck or discussion.


In [ ]:
if raw_results and not classified_df.empty:
    # Scenarios where ALL loaded mechanisms were correct
    unanimous_correct_ids = scen_df[scen_df['type'] == 'unanimous_correct']['scenario_id'].tolist()

    # Among those, compare contribution vs free_debate transcripts (canonical contrast)
    MECH_A = 'contribution'
    MECH_B = 'free_debate'

    path_analysis = []

    compare_ids = [s for s in unanimous_correct_ids[:15]
                   if s in idx.get(MECH_A, {}) and s in idx.get(MECH_B, {})]

    print(f"Comparing {MECH_A} vs {MECH_B} on {len(compare_ids)} unanimously-correct scenarios …")

    PATH_SYSTEM = (
        "You compare two deliberation transcripts where BOTH groups reached the correct answer.\n"
        "Determine whether the paths to the answer differed meaningfully.  Return JSON:\n"
        "{\n"
        "  \"same_path\": true/false,\n"
        "  \"mech_a_decisive_round\": <int or null — round decisive feature first mentioned>,\n"
        "  \"mech_b_decisive_round\": <int or null>,\n"
        "  \"mech_a_strategy\": \"<one phrase: e.g. 'explicit incentive', 'natural norm', 'lucky late'>\",\n"
        "  \"mech_b_strategy\": \"<one phrase>\",\n"
        "  \"notes\": \"<1-2 sentences>\"\n"
        "}\nNo markdown, only JSON."
    )

    for sid in compare_ids:
        wa = idx[MECH_A][sid]
        wb = idx[MECH_B][sid]
        ta = (extract_transcript(wa) or '')[:2500]
        tb = (extract_transcript(wb) or '')[:2500]
        user_msg = (
            f"Scenario {sid}. Correct answer: {_find_key(wa, ['correct_answer','answer','label'], '?')}\n"
            f"--- {MECH_A} TRANSCRIPT ---\n{ta}\n\n"
            f"--- {MECH_B} TRANSCRIPT ---\n{tb}"
        )
        try:
            resp = client.messages.create(
                model='claude-sonnet-4-20250514',
                max_tokens=300,
                system=PATH_SYSTEM,
                messages=[{'role': 'user', 'content': user_msg}],
            )
            text = resp.content[0].text.strip()
            text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.DOTALL).strip()
            result = json.loads(text)
            result['scenario_id'] = sid
            path_analysis.append(result)
            time.sleep(0.3)
        except Exception as e:
            print(f"  ⚠  {sid}: {e}")

    if path_analysis:
        path_df = pd.DataFrame(path_analysis)
        print(f"\nPath analysis complete for {len(path_df)} scenarios.")
        print(f"  Same path   : {path_df['same_path'].sum()}")
        print(f"  Diff path   : {(~path_df['same_path']).sum()}")

        # Round of decisive surfacing comparison
        path_df['decisive_round_diff'] = path_df['mech_b_decisive_round'] - path_df['mech_a_decisive_round']
        print(f"\nMean round of decisive surfacing:")
        print(f"  {MECH_A}: {path_df['mech_a_decisive_round'].mean():.2f}")
        print(f"  {MECH_B}: {path_df['mech_b_decisive_round'].mean():.2f}")
        print(f"  Diff (B-A):  {path_df['decisive_round_diff'].mean():.2f}  ")
        print(f"  (positive = {MECH_B} surfaces decisive info later)")

        # Show cases where paths diverged
        diff_cases = path_df[~path_df['same_path']]
        if not diff_cases.empty:
            print(f"\nDivergent-path examples:")
            for _, row in diff_cases.head(4).iterrows():
                print(f"  Scenario {row['scenario_id']}: {MECH_A}='{row['mech_a_strategy']}', {MECH_B}='{row['mech_b_strategy']}' — {row['notes']}")
    else:
        print("No path analysis results (possibly no raw transcript data).")
else:
    print("Skipping — requires raw trajectory data and classified results.")


### 8.10  Error Analysis Summary

Synthesise the qualitative findings into a concise narrative for the paper.


In [ ]:
if not classified_df.empty:
    total      = len(classified_df)
    top_fail   = classified_df['failure_pattern'].value_counts().index[0]
    top_succ   = classified_df['success_pattern'].value_counts().index[0]
    top_fail_n = classified_df['failure_pattern'].value_counts().iloc[0]
    top_succ_n = classified_df['success_pattern'].value_counts().iloc[0]

    print("="*72)
    print("ERROR ANALYSIS SUMMARY")
    print("="*72)
    print(f"\nTotal divergent pairs analysed : {total}")
    print(f"\nTop failure mode  : {top_fail} ({top_fail_n}/{total} = {top_fail_n/total:.0%})")
    print(f"Top success mode  : {top_succ} ({top_succ_n}/{total} = {top_succ_n/total:.0%})")

    # Per-mechanism failure profile
    print("\nMechanism failure profile:")
    for mech in all_metrics:
        fails = classified_df[classified_df['loser'] == mech]
        if fails.empty:
            continue
        dominant = fails['failure_pattern'].value_counts().index[0]
        print(f"  {mech:<35} n={len(fails):2d}  dominant failure: {dominant}")

    print("\nKey qualitative finding:")
    print("  Examine the pattern counts and mechanism heatmap above to articulate")
    print("  whether free-riding (FREE_RIDE_WAIT / FREE_RIDE_PIGGYBACK) or")
    print("  cognitive failures (MISLEADING_DOMINANCE, PREMATURE_CONSENSUS)")
    print("  dominate — and whether they cluster by mechanism type.")
    print()

    # Export for paper appendix
    export_cols = ['scenario_id', 'winner', 'loser', 'failure_pattern',
                   'success_pattern', 'failure_quote', 'success_quote', 'notes']
    export_df = classified_df[[c for c in export_cols if c in classified_df.columns]]
    export_df.to_csv('error_analysis_cases.csv', index=False)
    print("Per-case results saved to error_analysis_cases.csv")
else:
    print("No classified data available for summary.")
